# 🛠️ Git & Dev Tools for AI Engineering

---

## 1. Overview

Writing AI models and algorithms is only one part of an AI Engineer's daily workflow. Collaborating with teams, managing model configurations, keeping track of prompt iterations, and automating validation tests are equally critical. 

This notebook covers the development tools and operations layer of AI engineering:
- **Git & GitHub**: The industry standard for version control. We track prompt revisions, config files, and code. We use workflows (branching, merging, rebasing) to collaborate safely.
- **Bash/Shell Scripting**: The command line is the interface for cloud VMs, GPU clusters, and model execution environments. Writing robust shell scripts allows us to automate virtual environment setups, download weights (e.g., HuggingFace models), and run local inference servers.
- **GitHub Actions (CI/CD)**: Automating testing via workflows to guarantee that code formats, lint rules, and unit tests pass on every pull request.

We will implement these tools programmatically inside a sandboxed environment to show exactly how they function under the hood.

## 2. Learning Objectives

By the end of this notebook, you will be able to:

- Explain the core stages of Git (Working Directory, Staging Area, Local Repository, Remote).
- Execute Git branching, merging, and rebasing, and understand when to use each.
- Resolve Git merge conflicts programmatically and explain their structural markers.
- Write robust Bash/Shell scripts with error handling (`set -e`, `set -o pipefail`), arguments, and variables.
- Design a GitHub Actions workflow YAML file for continuous integration (CI) tests.
- Automate everyday AI engineering operational tasks, such as model weight verification and pipeline validation.

## 3. Imports

Although this notebook covers command-line and version control tools, we will use Python's built-in `subprocess`, `os`, and `pathlib` libraries to orchestrate these tools programmatically inside a sandboxed folder. This ensures every command runs cleanly and is fully reproducible.

In [19]:
import os
import sys
import shutil
import tempfile
import subprocess
from pathlib import Path

print(f"Python Version: {sys.version.split()[0]}")

Python Version: 3.12.13


## 4. Configuration

We will verify that `git` and a compatible shell (`bash` or standard shell command interpreters) are available on your system.

In [20]:
def check_tool_installed(tool_name: str) -> bool:
    """Checks if a command-line tool is available on the system PATH."""
    return shutil.which(tool_name) is not None

git_installed = check_tool_installed("git")
bash_installed = check_tool_installed("bash")

print(f"Git is installed: {git_installed}")
print(f"Bash is installed: {bash_installed}")

if git_installed:
    git_version = subprocess.run(["git", "--version"], capture_output=True, text=True)
    print(f"Git version details: {git_version.stdout.strip()}")

Git is installed: True
Bash is installed: True
Git version details: git version 2.34.1


## 5. Theory & Key Concepts

### 5.1 Git Architecture & Lifecycle

Git is not just a backup tool; it is a content tracker that manages state changes in trees of commits. Under the hood, Git maps content to a key-value store using SHA-1 hashes.

A file in a Git directory moves through four primary zones:

1. **Working Directory**: Your local directory where you modify code. Files here are *untracked* or *modified*.
2. **Staging Area (Index)**: A file that tracks what will go into your next commit. You add files here using `git add`.
3. **Local Repository**: The `.git` directory on your machine containing all historical commits. Created when you run `git commit`.
4. **Remote Repository**: A shared host (e.g., GitHub, GitLab) where commits are uploaded using `git push` so other team members can download them using `git fetch` or `git pull`.

```
               +------------+              +------------+              +------------+
               |  Working   |   git add    |  Staging   | git commit   |   Local    |
               | Directory  | -----------> |    Area    | -----------> | Repository |
               +------------+              +------------+              +------------+
                     ^                           |                           |
                     |                           |                           | git push
                     |       git checkout        |                           v
                     +---------------------------+                     +------------+
                                                                       |   Remote   |
                                                                       | Repository |
                                                                       +------------+
```

### 5.2 Merging vs. Rebasing

When combining changes from a feature branch back into a main branch, Git offers two main strategies:

#### 1. Merging (`git merge`)
- Performs a 3-way merge between the two branches and their common ancestor.
- Creates a dedicated **Merge Commit**.
- **Pros**: Preserves the complete, chronological history of when changes actually occurred.
- **Cons**: Commit graphs can become cluttered and hard to read ("railroad track" histories).

#### 2. Rebasing (`git rebase`)
- Re-applies commits from your current branch on top of another base branch.
- Rewrites history by creating brand new commits with new hashes.
- **Pros**: Creates a perfectly linear history, clean and easy to follow.
- **Cons**: Rewrites history. **Never rebase commits that have been pushed to public/shared branches**.

```
Initial State:
      A---B  (main)
           \
            C---D  (feature)

After Merge:
      A---B-------M  (main, contains merge commit M)
           \     /
            C---D  (feature)

After Rebase:
      A---B  (main)
           \
            C'---D' (feature, linear re-applied history)
```

### 5.3 Shell Scripting Best Practices

Bash is the standard glue language of systems engineering. In AI pipelines, shell scripts coordinate downloading datasets, initializing CUDA modules, running container setups, and starting servers. When writing scripts, defensive programming is key:

- `set -e`: Exit immediately if any command exits with a non-zero status.
- `set -u`: Exit if a script attempts to use an uninitialized variable.
- `set -o pipefail`: Ensure that pipeline commands (e.g., `cmd1 | cmd2`) return the exit code of the *first command to fail*, rather than the status of the final command.

Always wrap environment variables in double quotes (`"$VAR"`) to prevent issues with whitespace or empty parameters.

## 6. Implementation: Git Operations Sandbox

We will set up a sandboxed temporary git repository on your filesystem, make commits, simulate a merge conflict, and resolve it.

In [21]:
# Create a temporary sandboxed directory
temp_dir = tempfile.mkdtemp(prefix="git_sandbox_")
sandbox_path = Path(temp_dir)
print(f"Git Sandbox initialized at: {sandbox_path}")

Git Sandbox initialized at: /tmp/git_sandbox_rldaxxw0


In [22]:
def run_git(args: List[str], cwd: Path = sandbox_path) -> str:
    """Helper to run Git commands in our sandbox."""
    # Configure mock user details so git commands execute cleanly
    env = os.environ.copy()
    env["GIT_AUTHOR_NAME"] = "AI Engineer Client"
    env["GIT_AUTHOR_EMAIL"] = "client@ai-engineer.com"
    env["GIT_COMMITTER_NAME"] = "AI Engineer Client"
    env["GIT_COMMITTER_EMAIL"] = "client@ai-engineer.com"
    
    result = subprocess.run(
        ["git"] + args,
        cwd=str(cwd),
        capture_output=True,
        text=True,
        env=env
    )
    if result.returncode != 0:
        return f"ERROR: {result.stderr.strip()}"
    return result.stdout.strip()

# 1. Initialize git
print(run_git(["init", "-b", "main"]))

# 2. Configure local defaults to bypass global config rules
run_git(["config", "user.name", "AI Engineer Client"])
run_git(["config", "user.email", "client@ai-engineer.com"])

Initialized empty Git repository in /tmp/git_sandbox_rldaxxw0/.git/


''

### 6.1 Making a Commit on Main

Let's create a prompt template file and commit it.

In [23]:
prompt_file = sandbox_path / "prompt_template.txt"
with open(prompt_file, "w") as f:
    f.write("You are a helpful assistant.\nAnswer the following question: {question}\n")

print("--- Git Status ---")
print(run_git(["status"]))

print("\n--- Staging File ---")
run_git(["add", "prompt_template.txt"])
print(run_git(["status"]))

print("\n--- Committing ---")
print(run_git(["commit", "-m", "Initial commit with base prompt template"]))

print("\n--- Git Log ---")
print(run_git(["log", "--oneline"]))

--- Git Status ---
On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	prompt_template.txt

nothing added to commit but untracked files present (use "git add" to track)

--- Staging File ---
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   prompt_template.txt

--- Committing ---
[main (root-commit) fab9417] Initial commit with base prompt template
 1 file changed, 2 insertions(+)
 create mode 100644 prompt_template.txt

--- Git Log ---
fab9417 Initial commit with base prompt template


### 6.2 Creating a Branch & Conflict

We will branch out to `optimize-prompt` and modify our template. In parallel, we will modify the template on `main` to create a merge conflict.

In [24]:
# 1. Create and switch to new branch
print("Create branch:", run_git(["checkout", "-b", "optimize-prompt"]))

# 2. Edit template file on branch
with open(prompt_file, "w") as f:
    f.write("System Instruction: You are a strict Python developer.\nUser Question: {question}\nCode: ")

run_git(["add", "prompt_template.txt"])
print("Commit on branch:", run_git(["commit", "-m", "Update template for code generation"]))

# 3. Switch back to main
print("Switch back to main:", run_git(["checkout", "main"]))

# 4. Edit template file on main differently
with open(prompt_file, "w") as f:
    f.write("You are a concise data science tutor.\nExplain: {question}\nSummary: ")

run_git(["add", "prompt_template.txt"])
print("Commit on main:", run_git(["commit", "-m", "Modify template on main branch"]))

# 5. Check log tree
print("\nLog Tree:")
print(run_git(["log", "--all", "--graph", "--oneline"]))

Create branch: 
Commit on branch: [optimize-prompt 26c74bb] Update template for code generation
 1 file changed, 3 insertions(+), 2 deletions(-)
Switch back to main: 
Commit on main: [main 470d287] Modify template on main branch
 1 file changed, 3 insertions(+), 2 deletions(-)

Log Tree:
* 470d287 Modify template on main branch
| * 26c74bb Update template for code generation
|/  
* fab9417 Initial commit with base prompt template


### 6.3 Merging and Resolving the Conflict

Now, we will try to merge `optimize-prompt` into `main`. Git will alert us to a conflict because the same file lines were modified on both branches.

In [25]:
print("Attempting merge:")
merge_output = run_git(["merge", "optimize-prompt"])
print(merge_output)

print("\nFile content containing conflict markers:")
with open(prompt_file, "r") as f:
    print(f.read())

Attempting merge:
ERROR: 

File content containing conflict markers:
<<<<<<< HEAD
You are a concise data science tutor.
Explain: {question}
Summary: 
System Instruction: You are a strict Python developer.
User Question: {question}
Code: 
>>>>>>> optimize-prompt



#### Understanding conflict markers:
- `<<<<<<< HEAD`: The code in the current checked-out branch (`main`).
- `=======`: The divider between the two versions.
- `>>>>>>> optimize-prompt`: The code in the incoming branch being merged.

Let's write code to resolve the conflict by keeping elements of both or picking the system prompt from the optimize branch, clean the markers, and complete the merge commit.

In [26]:
# Resolve the conflict by rewriting the file
resolved_content = """System Instruction: You are a strict Python developer and concise data science tutor.
User Question: {question}
Code/Summary: """

with open(prompt_file, "w") as f:
    f.write(resolved_content)

# Mark conflict as resolved and finish merge
print("Add resolved file:", run_git(["add", "prompt_template.txt"]))
print("Commit merge:", run_git(["commit", "-m", "Resolve merge conflict combining both templates"]))

# Check updated log tree
print("\nLog Tree:")
print(run_git(["log", "--graph", "--oneline"]))

Add resolved file: 
Commit merge: [main 1b79316] Resolve merge conflict combining both templates

Log Tree:
*   1b79316 Resolve merge conflict combining both templates
|\  
| * 26c74bb Update template for code generation
* | 470d287 Modify template on main branch
|/  
* fab9417 Initial commit with base prompt template


### Cleanup Sandbox

We clean up the temporary directory to ensure a clean local filesystem.

In [27]:
shutil.rmtree(temp_dir)
print("Sandbox successfully cleaned up!")

Sandbox successfully cleaned up!


## 7. Implementation: Bash Scripting for AI Engineers

AI Engineers frequently need to initialize environments or download weights. We will write a Bash script that executes inside a sandbox and handles basic check routines.

### 7.1 Writing the Bash Script

We will write a shell script named `setup_model.sh` that checks if python is installed, validates a dummy model config file exists, and writes out a status metadata file.

In [28]:
bash_script_content = """#!/usr/bin/env bash

# Enable strict exit rules
set -e
set -u
set -o pipefail

echo "=== Starting AI Environment Verification ==="

# 1. Check if python is available
if command -v python3 >/dev/null 2>&1; then
    PYTHON_CMD="python3"
elif command -v python >/dev/null 2>&1; then
    PYTHON_CMD="python"
else
    echo "Error: Python is not installed!" >&2
    exit 1
fi

echo "Using python command: $PYTHON_CMD"
PYTHON_VERSION=$($PYTHON_CMD --version)
echo "Python version: $PYTHON_VERSION"

# 2. Mock model config setup
CONFIG_FILE="model_config.json"
if [ ! -f "$CONFIG_FILE" ]; then
    echo "Configuration file missing. Generating a template..."
    echo '{"model": "phi-4", "max_tokens": 1024, "temperature": 0.2}' > "$CONFIG_FILE"
else
    echo "Configuration file found!"
fi

# 3. Write metadata logs
echo "Writing setup details..."
echo "setup_status=completed" > setup_status.env
echo "timestamp=$(date)" >> setup_status.env

echo "=== Setup Completed Successfully ==="
"""

# Save script locally
script_path = Path("setup_model.sh")
with open(script_path, "w", newline="\n") as f:
    f.write(bash_script_content)

print(f"Script written to {script_path.absolute()}")

Script written to /content/setup_model.sh


### 7.2 Executing the Bash Script

Let's execute the script. On Unix environments, we must mark the script as executable (`chmod +x`). On Windows environments, if Bash is not installed, we can fall back to executing standard python logic or using the local shell interpreter.

In [29]:
if check_tool_installed("bash"):
    # Set execute permission on Unix systems
    subprocess.run(["chmod", "+x", "setup_model.sh"])
    
    # Run script
    result = subprocess.run(["bash", "setup_model.sh"], capture_output=True, text=True)
    print("--- STDOUT ---")
    print(result.stdout)
    print("--- STDERR ---")
    print(result.stderr)
    
    # View generated files
    if os.path.exists("setup_status.env"):
        with open("setup_status.env") as f:
            print("\n--- Generated Environment Variables File ---")
            print(f.read())
else:
    print("Skipping bash execution since bash command interpreter is not available on this machine.")

--- STDOUT ---
=== Starting AI Environment Verification ===
Using python command: python3
Python version: Python 3.12.13
Configuration file missing. Generating a template...
Writing setup details...
=== Setup Completed Successfully ===

--- STDERR ---


--- Generated Environment Variables File ---
setup_status=completed
timestamp=Mon Jun  8 10:52:27 PM UTC 2026



### Clean Up Generated Files

In [30]:
for file in ["setup_model.sh", "model_config.json", "setup_status.env"]:
    if os.path.exists(file):
        os.remove(file)
print("Temporary shell scripts and configs deleted successfully.")

Temporary shell scripts and configs deleted successfully.


## 8. Continuous Integration: GitHub Actions (CI)

Continuous Integration (CI) guarantees that modifications to code or models do not break your project. In GitHub, CI pipelines are configured using YAML files located in `.github/workflows/`.

### 8.1 Example GitHub Workflow: `ci.yml`

Below is a production-quality workflow config designed to check python code quality and run tests on every Push or Pull Request targeting `main`:

```yaml
name: Continuous Integration

on:
  push:
    branches: [ main ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]

    steps:
    - name: Checkout repository code
      uses: actions/checkout@v4

    - name: Set up Python ${{ matrix.python-version }}
      uses: actions/setup-python@v5
      with:
        python-version: ${{ matrix.python-version }}
        cache: 'pip'

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install ruff pytest
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Lint and Format Check with Ruff
      run: |
        # Run static analysis and formatting validation
        ruff check .
        ruff format --check .

    - name: Run Unit Tests with pytest
      run: |
        pytest
```

## 9. Guided Exercises

### Exercise 1: Git Stash & Branch Context Switch

Explain what commands you should run to perform a context switch when you have dirty (uncommitted) changes in your working directory on `main`, but need to instantly switch to a branch called `hotfix-api` without losing your uncommitted work or committing unfinished code.

#### Solution:
1. Save your uncommitted changes to git's temporary stash storage:
   ```bash
   git stash
   ```
2. Switch to the target branch:
   ```bash
   git checkout hotfix-api
   ```
3. Perform your edits on the hotfix branch, commit them, and check out `main` again:
   ```bash
   git checkout main
   ```
4. Restore your stashed changes to your working directory:
   ```bash
   git stash pop
   ```

### Exercise 2: Scripting a File Type Counter

Implement a short Python script or Bash script snippet that scans a folder (passed as the first argument) and lists the count of files ending with `.json` vs. `.py`.

In [31]:
def count_files(target_dir: str) -> Dict[str, int]:
    counts = {".py": 0, ".json": 0}
    path = Path(target_dir)
    if not path.exists():
        return counts
    
    for file in path.glob("**/*"):
        if file.suffix in counts:
            counts[file.suffix] += 1
            
    return counts

# Test counter on the current directory
print(count_files("."))

{'.py': 0, '.json': 2}


## 10. Challenge Problems

### Challenge 1: Automating Model Checkpoint Verification

AI model weights can be corrupted during download. Write a Python script (or Bash script if you are on a Unix system) that:
1. Scans a directory for any files matching `.bin` or `.safetensors` (simulate these by creating tiny dummy files).
2. Validates that the file size is larger than a threshold (e.g., 100 bytes).
3. Computes the SHA-256 hash of the file.
4. Writes out a `manifest.json` catalog containing the name, path, file size, and calculated SHA-256 hash for every valid model file found.

In [32]:
import json
import hashlib

def verify_model_checkpoints(target_folder: str) -> str:
    folder = Path(target_folder)
    folder.mkdir(exist_ok=True)
    
    # 1. Create a dummy model file for demonstration
    dummy_file = folder / "llama-mini.safetensors"
    with open(dummy_file, "wb") as f:
        f.write(b"Model weights placeholder " * 50)  # 1250 bytes
        
    # 2. Scan and verify
    manifest = {}
    for filepath in folder.glob("*"):
        if filepath.suffix in [".bin", ".safetensors"]:
            size = filepath.stat().st_size
            if size > 100:
                # Calculate SHA-256
                sha256 = hashlib.sha256()
                with open(filepath, "rb") as f:
                    for chunk in iter(lambda: f.read(4096), b""):
                        sha256.update(chunk)
                
                manifest[filepath.name] = {
                    "path": str(filepath.absolute()),
                    "size_bytes": size,
                    "sha256": sha256.hexdigest()
                }
                
    # 3. Write manifest.json
    manifest_path = folder / "manifest.json"
    with open(manifest_path, "w") as f:
        json.dump(manifest, f, indent=4)
        
    # Clean up model file but keep manifest structure representation
    if dummy_file.exists():
        dummy_file.unlink()
        
    with open(manifest_path, "r") as f:
        output = f.read()
    
    if manifest_path.exists():
        manifest_path.unlink()
        
    return output

print(verify_model_checkpoints("verify_sandbox"))
# Clean up folder
if os.path.exists("verify_sandbox"):
    shutil.rmtree("verify_sandbox")

{
    "llama-mini.safetensors": {
        "path": "/content/verify_sandbox/llama-mini.safetensors",
        "size_bytes": 1300,
        "sha256": "f5f1565f7246087e5ce88cc48519c4ce729561e8db8d1672691b65169c26e5e2"
    }
}


### Challenge 2: Pull Request Quality Control Action

Write a YAML file for a GitHub Action that enforces that **no commit contains prompt_template changes without accompanying unit test file changes**. Research how this can be implemented in a shell step using `git diff --name-only origin/main` to inspect changed file paths.

In [33]:
# Propose the workflow runner step here
pr_check_snippet = """
    - name: Enforce Test Updates
      run: |
        # Fetch main to compare diffs
        git fetch origin main
        
        # Get list of changed files
        CHANGED_FILES=$(git diff --name-only origin/main)
        
        # Check if templates changed
        if echo "$CHANGED_FILES" | grep -q "prompt_template.txt"; then
          # Enforce that tests also changed
          if ! echo "$CHANGED_FILES" | grep -q "test_"; then
            echo "Error: Detected modifications to prompt_template.txt without updates to test suite files!" >&2
            exit 1
          fi
        fi
        echo "All quality checks passed!"
"""
print(pr_check_snippet)


    - name: Enforce Test Updates
      run: |
        # Fetch main to compare diffs
        git fetch origin main
        
        # Get list of changed files
        CHANGED_FILES=$(git diff --name-only origin/main)
        
        # Check if templates changed
        if echo "$CHANGED_FILES" | grep -q "prompt_template.txt"; then
          # Enforce that tests also changed
          if ! echo "$CHANGED_FILES" | grep -q "test_"; then
            echo "Error: Detected modifications to prompt_template.txt without updates to test suite files!" >&2
            exit 1
          fi
        fi
        echo "All quality checks passed!"



## 11. Further Reading

- **Pro Git Book**: [https://git-scm.com/book/en/v2](https://git-scm.com/book/en/v2) - The ultimate guide to Git architecture and usage.
- **Advanced Bash-Scripting Guide**: [https://tldp.org/LDP/abs/html/](https://tldp.org/LDP/abs/html/) - Detailed explanations of shell options and operations.
- **GitHub Actions Documentation**: [https://docs.github.com/en/actions](https://docs.github.com/en/actions) - Syntaxes, caches, and custom runner environments.

---

**[← Previous: 06 — Pandas & Polars](06-pandas-polars-data.ipynb)** · **[Next: Phase 2 →](../02-ml-fundamentals/)**